# Parte 1: dataset y preprocesamiento

**Notebook en Colab:** LO DEJAMOS PENDIENTE PARA CUANDO LO SUBAMOS

## Equipo
- Alumno 1 : Lucas Ramirez 
- Alumno 2 : Valentín Lujambio

## 0. Configuración

El código reutilizable vive en `src/lib/` y se importa desde acá, desde la notebook 2 y desde el backend. Las semillas se fijan una sola vez.

**En Colab:** clonar el fork (`!git clone ... && %cd tuia-face-recognition-app`), instalar `!pip install -r requirements.txt` y ejecutar desde la raíz del repo.

In [ ]:
import random
import shutil
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from lib.dataset.sources import FolderDataset, find_dataset_root, strip_prefix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATA_DIR = ROOT / "data"          # data/* está en .gitignore: las imágenes no se suben
RAW_DIR = DATA_DIR / "raw" / "pins"

## 2.1 Elección del dataset

Elegimos **Pins Face Recognition** (Kaggle, `hereisburak/pins-face-recognition`): 105 celebridades y 17.534 imágenes recolectadas de Pinterest, una carpeta por persona. Los números exactos salen de la celda de carga.

| Opción | A favor | En contra |
|---|---|---|
| **Pins** (elegido) | Pocas personas con muchas fotos cada una, que es lo que necesita una CNN entrenada desde cero (modelo A): aprende a reconocer a alguien viendo muchas variaciones de esa misma persona. Alcanza para separar identidades enteras para open-set. | Pocas identidades. Fotos de celebridades: producción profesional, maquillaje, sesgo demográfico probable. |
| LFW | Protocolo oficial de pares; es el de la práctica de la cátedra. | La mayoría de las 5.749 personas tiene una sola foto. Con ≥ 20 fotos quedan ~62 personas y ~3.000 imágenes, muy desbalanceado. |
| CelebA / CASIA-WebFace | Muchas identidades y variedad. | Pesados; CASIA tiene ruido de etiquetas. |

**Relación con la app:** la app registra pocas personas con pocas fotos y reconoce caras nuevas. Lo que más importa es que el embedding generalice a identidades no vistas, y Pins permite medirlo separando identidades (2.7).

**Limitaciones y cómo pueden afectar:**
- **Dominio:** fotos de prensa y redes de celebridades. La app va a recibir fotos de celular o webcam, con peor luz y resolución, así que el rendimiento real puede ser menor que el medido.
- **Pocas identidades:** con ~105 personas, la verificación open-set se mide sobre pocas identidades y pocos pares impostores distintos, lo que hace las métricas a FAR bajo (1e-3) poco estables.
- **Sesgo demográfico:** se analiza en 2.8.
- **Etiquetas:** al ser recolectado de la web puede haber fotos mal etiquetadas o con otra persona; lo auditamos en 2.5.

**Licencia:** en Kaggle el dataset figura como **CC0: Public Domain** (dato de la API de Kaggle, `licenseName`), lo que permite el uso académico. La descripción aclara que las imágenes fueron recolectadas de Pinterest y recortadas (105 celebridades, 17.534 caras). Hay un matiz: la licencia CC0 la declara quien subió el dataset, pero los derechos de cada foto son de sus autores originales, y las personas fotografiadas no dieron consentimiento para este uso. Lo retomamos en 2.8.

### Descarga y carga

Se descarga con `kagglehub` (viene instalado en Colab; local: `pip install kagglehub`). Requiere un token de Kaggle (`~/.kaggle/kaggle.json` o las variables `KAGGLE_USERNAME` y `KAGGLE_KEY`). Alternativa manual: bajar el zip desde la página del dataset y descomprimirlo en `data/raw/pins/`.

`FolderDataset` lee cualquier dataset con una carpeta por identidad, así el resto de la notebook no depende de Pins. Las carpetas de Pins se llaman `pins_<Nombre>`: `strip_prefix` deja solo el nombre.

In [ ]:
if not RAW_DIR.exists():
    import kagglehub
    downloaded = Path(kagglehub.dataset_download("hereisburak/pins-face-recognition"))
    shutil.copytree(downloaded, RAW_DIR)

DATASET_ROOT = find_dataset_root(RAW_DIR)
dataset = FolderDataset(DATASET_ROOT, label_parser=strip_prefix("pins_")).to_frame()
print(f"Raíz del dataset: {DATASET_ROOT.relative_to(ROOT)}")
print(f"Identidades: {dataset['identity'].nunique()} | Imágenes: {len(dataset)}")
dataset.head()